# Email Campaign A/B Test

This project looks at a real email marketing experiment released by Kevin Hillstrom
for the MineThatData analytics challenge.

Customers were randomly split into three groups:

- Men's E-Mail
- Women's E-Mail
- No E-Mail

I want to find out whether either campaign actually improved visits, conversions
and customer spend compared with sending no email.


## 1. Load the data

The dataset contains 64,000 customers. If the CSV is not in the `data` folder yet,
run `python download_data.py` from the project root first.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import ttest_ind

REPO_ROOT = Path("..")
sys.path.append(str(REPO_ROOT))

from src.ab_test_utils import (
    balance_table,
    experiment_summary,
    proportion_test,
)

data = pd.read_csv(REPO_ROOT / "data" / "hillstrom.csv")
data.head()


In [ ]:
print("Rows and columns:", data.shape)
data.info()


In [ ]:
print("Missing values:")
print(data.isna().sum())

print("\nDuplicate rows:", data.duplicated().sum())


There is not much cleaning needed here, so I am keeping the original values rather
than changing the dataset just for the sake of having a cleaning section.


## 2. Check the experiment groups

Before comparing outcomes, I want to check the group sizes and whether the main
customer characteristics look reasonably similar across the randomised groups.


In [ ]:
data["segment"].value_counts()


In [ ]:
balance = balance_table(data)
balance.round(3)


The averages are close across the three groups, which is what I would expect from
random assignment. This does not prove perfect balance, but there is no obvious
difference that would make one group look fundamentally different before the email.


## 3. Compare the headline outcomes


In [ ]:
summary = experiment_summary(data)
summary.round(4)


In [ ]:
plot_data = summary[
    ["visit_rate", "conversion_rate"]
].mul(100)

plot_data.plot(kind="bar", figsize=(9, 5))
plt.ylabel("Rate (%)")
plt.xlabel("")
plt.title("Visit and conversion rates by experiment group")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


The email groups have higher visit and conversion rates than the no-email group.
The next step is to check whether those differences are large enough that I would
not put them down to normal sampling variation.


## 4. Statistical test — Men's email vs no email

Conversion is a yes/no outcome, so I use a two-proportion z-test.


In [ ]:
mens = data[data["segment"] == "Mens E-Mail"]
womens = data[data["segment"] == "Womens E-Mail"]
control = data[data["segment"] == "No E-Mail"]

mens_conversion = proportion_test(
    control["conversion"],
    mens["conversion"],
)

pd.Series(mens_conversion)


In [ ]:
print(f"Control conversion: {mens_conversion['control_rate']:.2%}")
print(f"Men's email conversion: {mens_conversion['treatment_rate']:.2%}")
print(f"Absolute difference: {mens_conversion['absolute_difference']:.2%}")
print(f"Relative lift: {mens_conversion['relative_lift']:.1%}")
print(f"P-value: {mens_conversion['p_value']:.3g}")
print(
    "95% CI:",
    f"{mens_conversion['ci_low']:.2%} to {mens_conversion['ci_high']:.2%}"
)


## 5. Women's email vs no email


In [ ]:
womens_conversion = proportion_test(
    control["conversion"],
    womens["conversion"],
)

pd.Series(womens_conversion)


## 6. Revenue per customer

Spend is highly skewed because most customers spend nothing, but average revenue
per customer is still commercially useful. I compare both the observed averages
and a Welch t-test, which does not assume the two groups have equal variance.


In [ ]:
revenue_summary = (
    data.groupby("segment")["spend"]
    .agg(["count", "mean", "median", "sum"])
)

revenue_summary.round(2)


In [ ]:
mens_spend_test = ttest_ind(
    mens["spend"],
    control["spend"],
    equal_var=False,
)

womens_spend_test = ttest_ind(
    womens["spend"],
    control["spend"],
    equal_var=False,
)

print("Men's email vs control p-value:", mens_spend_test.pvalue)
print("Women's email vs control p-value:", womens_spend_test.pvalue)


## 7. Does the result differ by customer type?

The overall experiment tells me whether the campaign worked on average. I also want
to see whether the effect looks different for new vs existing customers and across
the customer's previous purchase channel.


In [ ]:
customer_type_summary = (
    data.assign(
        customer_type=data["newbie"].map(
            {1: "New customer", 0: "Existing customer"}
        )
    )
    .groupby(["customer_type", "segment"])
    .agg(
        customers=("segment", "size"),
        conversion_rate=("conversion", "mean"),
        revenue_per_customer=("spend", "mean"),
    )
    .reset_index()
)

customer_type_summary.round(4)


In [ ]:
channel_summary = (
    data.groupby(["channel", "segment"])
    .agg(
        customers=("segment", "size"),
        conversion_rate=("conversion", "mean"),
        revenue_per_customer=("spend", "mean"),
    )
    .reset_index()
)

channel_summary.round(4)


## 8. Conclusion

The experiment is useful because treatment was randomly assigned, so the comparison
is much stronger than simply observing customers who happened to receive different
marketing.

My final decision would be based on three things together:

1. whether conversion improved,
2. whether revenue per customer improved,
3. whether the confidence interval is large enough to matter commercially.

I would also be careful not to treat exploratory subgroup differences as separate
confirmed experiments without further testing.
